# Web Scraping with Python

Some websites can contain a very large amount of invaluable data.

Stock prices, product details, sports stats, company contacts, you name it.

If you wanted to access this information, you’d either have to use whatever format the website uses or copy-paste the information manually into a new document. Here’s where web scraping can help.

##What is Web Scraping?
Web scraping refers to the extraction of data from a website. This information is collected and then exported into a format that is more useful for the user. Be it a spreadsheet or an API.

Although web scraping can be done manually (eg. copy-paste), in most cases, automated tools are preferred when scraping web data as they can be less costly and work at a faster rate.

But in most cases, web scraping is not a simple task. Websites come in many shapes and forms, as a result, web scrapers vary in functionality and features.

## How do Web Scrapers Work?
Automated web scrapers work in a rather simple but also complex way. After all, websites are built for humans to understand, not machines.

First, the web scraper will be given one or more URLs to load before scraping. The scraper then loads the entire HTML code from the page in question. More advanced scrapers will render the entire website, including CSS and Javascript elements.

Then the scraper will either extract all the data on the page or specific data selected by the user before the project is run.

Ideally, the user will go through the process of selecting the specific data they want from the page. For example, you might want to scrape an Amazon product page for prices and models but are not necessarily interested in product reviews.

Lastly, the web scraper will output all the data that has been collected into a format that is more useful to the user.

Most web scrapers will output data to a CSV or Excel spreadsheet, while more advanced scrapers will support other formats such as JSON which can be used for an API.

Source: [parsehub.com](https://www.parsehub.com/blog/what-is-web-scraping/)

We'll be using three Python packages for this unit:
- `requests`
- `pandas`
- `BeautifulSoup` (`bs4`)

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

Ultimately, we'll use requests to download HTML content from the web, and BeautifulSoup to parse that content into pandas data frames.

First, though, it's worth noting that pandas has a built-in function that makes it easy to grab HTML tables from online.




## Using pandas `read_html()`

pandas' `read_html()` will convert all of the tables from a URL into a list of data frames (one for each `<table>` element). (see below).

![Screen Shot 2021-05-25 at 9.47.56 AM.png](../assets/pandas_read_html_screenshot.png)


Here's how it operates on the 2023 team stats on the NFL website.

In [ ]:
nfl = pd.read_html("https://www.nfl.com/stats/team-stats/offense/passing/2023/reg/all")
nfl

We can access the elements of that list using square brackets.

In [ ]:
teams_2023 = nfl[0]
teams_2023.head()

### Using `read_html()` in a loop

Because the URL is structured very simply - the only thing that changes is the year in each URL - it would not be hard to write a loop that would gather several years for us at once.

In [ ]:
for year in range(2019, 2024):
  print(year)

In [ ]:
for year in range(2019, 2024):
  url = "https://www.nfl.com/stats/team-stats/offense/passing/" + str(year) + "/reg/all"
  print(url)

In [ ]:
for year in range(2019, 2024):
  url = "https://www.nfl.com/stats/team-stats/offense/passing/" + str(year) + "/reg/all"

  # read_html returns a list of data frames
  stats = pd.read_html(url)

  # extract the main data frame from the list
  stats = stats[0]

  # add a year variable to the data frame
  stats["year"] = year
  print(stats.head())

Let's do the same thing, but save the results to a list rather than printing them.

In [ ]:
nfl = []
for year in range(2019, 2024):
  url = "https://www.nfl.com/stats/team-stats/offense/passing/" + str(year) + "/reg/all"
  stats = pd.read_html(url)
  stats = stats[0]
  stats["year"] = year
  nfl.append(stats)

nfl

We've got a list of data frames - if we'd like to combine them, we can do that with `pd.concat()`

In [ ]:
nfl_combined = pd.concat(nfl, ignore_index=True)

nfl_combined

From there, we could perform any cleanup and analyses we'd like!

## Using requests and Beautiful Soup

Not all pieces of information that we'd like will always be present in a `<table>` HTML element. Sometimes we'll want to construct our own datasets using data that might have some structure (thanks to HTML) but doesn't necessarily exist in a table yet.

This will happen in two steps:


1.   Download an HTML page using requests
2.   Parse the HTML using Beautiful Soup

We're going to be using [Books to Scrape](http://books.toscrape.com/index.html), a site that intentionally makes itself available to teach and test web scraping. Following along with some of [this helpful blog post](https://towardsdatascience.com/an-introduction-to-web-scraping-with-python-a2601e8619e5), we will look to scrape the following information about the books on this page:
- title
- price
- availability
- rating
- url

### Downloading a web page

The first step is to use requests to obtain the HTML for a page.

In [ ]:
# Specify the URL we want to access
url = "http://books.toscrape.com/index.html"

# Download the HTML
page = requests.get(url)

# Check to see if it downloaded correctly by printing it as text
page.text

### Parsing the page

That seems like valid HTML, though it's a bit ugly. Let's parse this with Beautiful Soup and turn it into something a bit easier to read. We'll just print out the first few hundred characters so that it doesn't take over our screens.

In [ ]:
soup = BeautifulSoup(page.text, "html.parser")
print(soup.prettify()[:750])

### Understanding HTML

This is where an understanding of the structure of web pages and HTML will come in handy. Here are the bullet points you should absolutely know to move forward:
- HTML is a plaintext markup language that tells browsers how to render a web page
- HTML elements are created with tags like `<table></table>` or `<a></a>` or `<div></div>` that tell your browser how to display parts of a web page
- Tags usually have an open and close that tell you where content begins and ends, e.g. `<p>Paragraph content goes here</p>`
- Elements/tags can have aspects like classes (repeatable within a page, often used for styling), id (unique within a page), and named attributes (provide additional information about an element)
  - That might look like:
    - `<a class="my-class" href="https://google.com">Google!</a>`, where `href` is an attribute and `"my-class"` is a class for this hyperlink.
- Elements can be hierarchical, and some are even meant to be. Here is an example of a table and how it would render (`tr` is table row, `th` is table header, and `td` is table data)
  
```
<table>
    <tr>
        <th>
            Header one!
        </th>
        <th>
            Header two!
        </th>
        <th>
            Header three!
        </th>
    </tr>
    <tr>
        <td>
            Cell one!
        </td>
        <td>
            Cell two!
        </td>
        <td>
            Cell three!
        </td>
    </tr>
</table>
```

--------------

| Header one!   | Header two!   | Header three! |
| ------------- |:-------------:| -------------:|
| Cell one!     | Cell two!     | Cell three!   |


-------------

Most scrapers, including Beautiful Soup, can collect elements based on class, id, attributes, or type (`a`, `p`, `table`, etc.). The pandas `read_html()` function we used looks for `table` elements, for example.

You can see the underlying HTML of a web page in most modern browsers by right-clicking and choosing "View Source," "Inspect Element," or similar options.




### Extracting Page Elements

Notice that in the HTML we downloaded, the element containing information for each book is called `<article>`. We can use that to extract information from those elements.

We'll use the `find()` method to get the first `article` element.

In [ ]:
print(soup.find("article").prettify())

Within that, we can see that there is information we're looking for. Let's think about how to extract it from one book, and then we can run something similar on each book on the page.

Let's start with the rating, which is a `p` element with a two classes: `star-rating` and `Three` (an element can have more than one class, and they are separated by spaces). We want to extract `Three`, but let's start by making sure we have the correct `p` element.

In [ ]:
book = soup.find("article")

In [ ]:
book.find("p", class_="star-rating")

That looks right! Let's use the `get()` method to retrieve the classes associated with that `p`.

In [ ]:
book.find("p", class_="star-rating").get("class")

Great! Now let's get the rating, which is the `[1]` element of that list.

In [ ]:
book.find("p", class_="star-rating").get("class")[1]

We'll save that code for later and try to get a few more elements.

In [ ]:
title = book.find("h3").find("a").get("title")
print("Title:", title)

rating = book.find("p", class_="star-rating").get("class")[1]
print("Rating:", rating)

# .text extracts the text from an element
# .strip() removes leading and trailing white space from a string
price = book.find("p", class_="price_color").text.strip()
print("Price:", price)

availability = book.find("p", class_="availability").text.strip()
print("Availability:", availability)

# We'll add the "http://books.toscrape.com/" before each url ourselves
url = book.find("h3").find("a").get("href")
print("URL before cleanup:", url)
url = "http://books.toscrape.com/" + url
print("URL after cleanup:", url)

### Looping Through Multiple Elements

We have the beginnings of a proper scraper there! Let's try looping over all of the books on this page using `findAll()` rather than `find()` for each article.

In [ ]:
books = soup.find_all("article")

This gives us a list of matching elements that we can iterate over using a version of our previous code. Let's loop over that list and save the outputs to a bunch of lists.

In [ ]:
titles = []
ratings = []
prices = []
availabilities = []
urls = []

for book in books:
  title = book.find("h3").find("a").get("title")
  titles.append(title)

  rating = book.find("p", class_="star-rating").get("class")[1]
  ratings.append(rating)

  price = book.find("p", class_="price_color").text.strip()
  prices.append(price)

  availability = book.find("p", class_="availability").text.strip()
  availabilities.append(availability)

  url = book.find("h3").find("a").get("href")
  url = "http://books.toscrape.com/" + url
  urls.append(url)

# Another way to do the above: more "pythonic" to use list comprehensions
# titles = [book.find("h3").find("a").get("title") for book in books]
# rating = [book.find("p", class_="star-rating").get("class")[1] for book in books]
# price = [book.find("p", class_="price_color").text.strip() for book in books]
# availability = [book.find("p", class_="availability").text.strip() for book in books]
# url = ["http://books.toscrape.com/" + book.find("h3").find("a").get("href") for book in books] 

print(titles)
print(ratings)
print(prices)
print(availabilities)
print(urls)

From there, we can easily turn those into a pandas data frame.

In [ ]:
book_info = pd.DataFrame({
    "title": titles,
    "rating": ratings,
    "price": prices,
    "availability": availabilities,
    "url": urls
})

book_info

Let's combine those into a function we can use later.

In [ ]:
def book_scraper(page_url):
  # Use requests to download the content
  page = requests.get(page_url)

  # Use BeautifulSoup to parse the HTML
  soup = BeautifulSoup(page.text, "html.parser")

  # Find all the books on the page
  books = soup.find_all("article")

  # Run our loop on each book like we did before
  titles = []
  ratings = []
  prices = []
  availabilities = []
  urls = []

  for book in books:
    title = book.find("h3").find("a").get("title")
    titles.append(title)

    rating = book.find("p", class_="star-rating").get("class")[1]
    ratings.append(rating)

    price = book.find("p", class_="price_color").text.strip()
    prices.append(price)

    availability = book.find("p", class_="availability").text.strip()
    availabilities.append(availability)

    url = book.find("h3").find("a").get("href")
    url = "http://books.toscrape.com/" + url
    urls.append(url)
      
  # as before, more pythonic to do list comprehensions, but maybe a bit harder to understand
  # titles = [book.find("h3").find("a").get("title") for book in books]
  # rating = [book.find("p", class_="star-rating").get("class")[1] for book in books]
  # price = [book.find("p", class_="price_color").text.strip() for book in books]
  # availability = [book.find("p", class_="availability").text.strip() for book in books]
  # url = ["http://books.toscrape.com/" + book.find("h3").find("a").get("href") for book in books] 

  book_info = pd.DataFrame({
    "title": titles,
    "rating": ratings,
    "price": prices,
    "availability": availabilities,
    "url": urls
  })

  return(book_info)



Now let's try running that on our main page to see how it works.

In [ ]:
first_page = book_scraper("http://books.toscrape.com/index.html")

first_page

### Crawling an Entire Site

That gets us all of the books from the first page! If we want to go through the entire catalog, however, we need to perform this same operation on multiple pages. Fortunately for us, the URLs of each page here follow a similar pattern after the first page:
- `http://books.toscrape.com/catalogue/page-{PAGE-NUMBER}.html`

We also know that there are 50 pages by looking at the bottom to see "Page ___ of 50." We can use this to construct a list of all of the URLS for catalog pages, starting with the landing page.

In [ ]:
catalog_urls = ["http://books.toscrape.com/index.html"]

for i in range(2, 51):
  new_url = "http://books.toscrape.com/catalogue/page-" + str(i) + ".html"
  catalog_urls.append(new_url)

catalog_urls

Thanks to the function we wrote before, we should be able to crawl through all of these pages to get information about all of the books. Let's try it!

In [ ]:
# Let's make an empty list to hold our data frames
every_book = []

# Use our book_scraper function to scrape each page
for url in catalog_urls:
  print("Scraping page", url)
  tmp = book_scraper(url)
  every_book.append(tmp)

# another alternative to for loop above, but don't get status printouts
# every_book = [bookscraper(url) for url in catalog_urls]

# Inspect the first two to see if it worked
every_book[1:2]

### Combining Results

Ok, we've got a list of data frames! Let's combine those together!

In [ ]:
df_final = pd.concat(every_book, ignore_index=True)

df_final

Now we have basic information about all 1000 books on that site!

If we want to learn more about them, we can also use the URLs we collected for each of the items to dig deeper.

## Other Resources

- [Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [More In-Depth Tutorial](https://towardsdatascience.com/an-introduction-to-web-scraping-with-python-a2601e8619e5)
- [Discussion on the Legality/Ethics of Scraping](https://benbernardblog.com/web-scraping-and-crawling-are-perfectly-legal-right/)
